# 05 — RAG para Código Especializado

**Módulo:** EAI_07 — IA Generativa  
**Submódulo:** 03_RAG  
**Ambiente:** `eai07` (Python 3.11)

---

## O que você vai aprender

- **Chunking por AST** — extrair funções e classes de arquivos `.py` usando a árvore sintática real do Python, não cortes por tamanho
- **Índice híbrido** — combinar chunks de código (`.py`) e documentação (`.md`) no mesmo índice FAISS com campo `tipo` nos metadados
- **Busca híbrida** — combinar score semântico (FAISS) com score léxico (BM25) para cobrir tanto intenção quanto nomes exatos
- **Assistente de código** — perguntas sobre o próprio projeto EAI_07 respondidas com contexto real

---

### Por que RAG especializado para código?

Código tem características que quebram o RAG genérico:

| Problema | Exemplo | Solução aqui |
|---|---|---|
| Nomes exatos importam | buscar `llm_factory` vs "fábrica de modelos" | BM25 (léxico) |
| Corte no meio de função quebra contexto | `def func():` separada do corpo | Chunking por AST |
| Código e docs falam do mesmo conceito | `llm_factory.py` + `AGENT_CONTEXT.md` | Índice híbrido |
| Embedding semântico sozinho perde keywords | `chat_stream` não é "fluxo de chat" | Score combinado |

## Setup

In [1]:
import sys, os, ast, pickle, time, math, re
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from openai import OpenAI
from dotenv import load_dotenv
from collections import Counter

sys.path.append(os.path.abspath('..'))
load_dotenv('../.env')

modelo_emb = SentenceTransformer('all-MiniLM-L6-v2')
llm = OpenAI(
    api_key=os.getenv('DEEPSEEK_API_KEY'),
    base_url='https://api.deepseek.com'
)
LLM_MODEL  = os.getenv('LLM_MODEL', 'deepseek-chat')
PROJETO_BASE = os.path.abspath('../..')
EAI07_BASE   = os.path.abspath('..')

print(f'LLM     : {LLM_MODEL}')
print(f'Projeto : {PROJETO_BASE}')
print(f'EAI07   : {EAI07_BASE}')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


LLM     : deepseek-chat
Projeto : C:\Users\Jorge Maques\Documents\Especialista_em_AI
EAI07   : C:\Users\Jorge Maques\Documents\Especialista_em_AI\EAI_07_AI_Generative


---
## 1. Chunking por AST

O módulo `ast` do Python parseia o código-fonte e retorna uma árvore sintática.  
Em vez de cortar por número de caracteres, extraímos **funções e classes completas** como unidades naturais.

Cada chunk de código carrega metadados ricos:
- `tipo_chunk`: `função` | `classe` | `módulo`
- `assinatura`: a linha `def ...` ou `class ...` completa
- `docstring`: se existir, extraída automaticamente
- `arquivo`: caminho relativo do `.py`

In [2]:
def extrair_chunks_ast(caminho_py: str, modulo_eai: str = '') -> list:
    """
    Parseia um arquivo .py com ast e retorna chunks por função/classe.
    Cada chunk é uma unidade sintática completa, não um corte arbitrário.
    """
    with open(caminho_py, 'r', encoding='utf-8') as f:
        source = f.read()
    
    try:
        tree = ast.parse(source)
    except SyntaxError as e:
        print(f'  [AVISO] Erro de sintaxe em {caminho_py}: {e}')
        return []

    linhas   = source.splitlines()
    arquivo  = os.path.relpath(caminho_py, PROJETO_BASE).replace('\\', '/')
    nome_arq = os.path.basename(caminho_py)
    chunks   = []

    def _extrair_no(no, tipo):
        """Extrai código, assinatura e docstring de um nó AST."""
        linha_ini = no.lineno - 1
        linha_fim = getattr(no, 'end_lineno', len(linhas))
        corpo     = '\n'.join(linhas[linha_ini:linha_fim])

        # Assinatura: primeira linha (def/class)
        assinatura = linhas[linha_ini].strip()

        # Docstring: primeiro nó do corpo se for Expr > Constant
        docstring = ''
        if (isinstance(no, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef))
                and no.body
                and isinstance(no.body[0], ast.Expr)
                and isinstance(no.body[0].value, ast.Constant)):
            docstring = no.body[0].value.value.strip()

        # chunk_busca: texto otimizado para embedding
        partes_busca = [f'[{modulo_eai} / {nome_arq}] {tipo}: {no.name}']
        if docstring:
            partes_busca.append(docstring[:200])
        partes_busca.append(assinatura)

        # chunk_contexto: código completo para o LLM
        contexto = f'# Arquivo: {arquivo}\n# {tipo}: {no.name}\n\n{corpo}'

        return {
            'chunk_busca'   : ' | '.join(partes_busca),
            'chunk_contexto': contexto,
            'titulo'        : no.name,
            'modulo'        : modulo_eai,
            'tipo'          : 'codigo',
            'tipo_chunk'    : tipo,
            'arquivo'       : arquivo,
            'assinatura'    : assinatura,
            'docstring'     : docstring,
            'linhas'        : (no.lineno, linha_fim),
        }

    for no in ast.walk(tree):
        if isinstance(no, (ast.FunctionDef, ast.AsyncFunctionDef)):
            chunks.append(_extrair_no(no, 'função'))
        elif isinstance(no, ast.ClassDef):
            chunks.append(_extrair_no(no, 'classe'))

    # Se não há funções/classes, indexa o arquivo inteiro como chunk de módulo
    if not chunks and source.strip():
        resumo = ' '.join(source.split()[:60])
        chunks.append({
            'chunk_busca'   : f'[{modulo_eai} / {nome_arq}] módulo: {resumo}',
            'chunk_contexto': f'# Arquivo: {arquivo}\n\n{source}',
            'titulo'        : nome_arq,
            'modulo'        : modulo_eai,
            'tipo'          : 'codigo',
            'tipo_chunk'    : 'módulo',
            'arquivo'       : arquivo,
            'assinatura'    : '',
            'docstring'     : '',
            'linhas'        : (1, len(linhas)),
        })

    return chunks


print('extrair_chunks_ast() definida.')

extrair_chunks_ast() definida.


In [3]:
# Demonstração: parseia llm_factory.py e mostra os chunks extraídos
factory_path = os.path.join(EAI07_BASE, 'shared', 'llm_factory.py')

if os.path.exists(factory_path):
    chunks_demo = extrair_chunks_ast(factory_path, 'EAI_07_AI_Generative')
    print(f'Chunks extraídos de llm_factory.py: {len(chunks_demo)}\n')
    for c in chunks_demo:
        print(f"  [{c['tipo_chunk']:8}] {c['titulo']:30} linhas {c['linhas']}")
        if c['docstring']:
            print(f"               docstring: {c['docstring'][:80]}...")
else:
    print(f'Arquivo não encontrado: {factory_path}')
    print('Ajuste o caminho conforme sua estrutura.')

Chunks extraídos de llm_factory.py: 10

  [função  ] _find_and_load_env             linhas (34, 44)
               docstring: Procura o .env subindo pelos diretórios pai....
  [função  ] chat                           linhas (56, 89)
               docstring: Envia uma mensagem para o LLM configurado no .env e retorna a resposta.

    Par...
  [função  ] chat_stream                    linhas (92, 113)
               docstring: Versão streaming do chat — gera tokens em tempo real.

    Exemplo:
        for ...
  [função  ] get_provider_info              linhas (116, 123)
               docstring: Retorna informações sobre o provider e modelo ativos....
  [função  ] _get_openai_client             linhas (128, 143)
               docstring: Instancia o client OpenAI-compatível conforme o provider....
  [função  ] _build_messages                linhas (146, 151)
  [função  ] _chat_openai_sdk               linhas (154, 162)
  [função  ] _stream_openai_sdk             linhas (165, 177)
  [fu

---
## 2. Índice Híbrido: Código + Documentação

O mesmo índice FAISS recebe dois tipos de chunk:

- `tipo: 'codigo'` — funções e classes extraídas por AST dos `.py`
- `tipo: 'doc'` — seções dos `AGENT_CONTEXT.md` (mesmo chunking do notebook 03)

O campo `tipo` no metadata permite filtrar depois da busca — ou misturar tudo e deixar o score decidir.

In [4]:
# ── Reutiliza chunking de docs dos notebooks anteriores ────────────────────
ENRIQUECIMENTO = {
    'regressão linear'       : 'regressão linear, ajuste de curva, reta, mínimos quadrados',
    'CNN'                    : 'CNN, rede convolucional, convolutional neural network',
    'LSTM'                   : 'LSTM, Long Short-Term Memory, células de memória',
    'deep learning'          : 'deep learning, aprendizado profundo, redes neurais profundas',
    'embeddings'             : 'embeddings, vetores de palavras, representação vetorial',
    'transformers'           : 'transformers, BERT, GPT, attention, mecanismo de atenção',
    'RAG'                    : 'RAG, Retrieval Augmented Generation, busca semântica',
    'function calling'       : 'function calling, tool calling, ferramentas, tools',
    'llm_factory'            : 'llm_factory, fábrica de LLM, provedor de linguagem, chat, chat_stream',
    'tool_runner'            : 'tool_runner, execução de ferramentas, function calling, DeepSeek tools',
    'FAISS'                  : 'FAISS, índice vetorial, busca por similaridade, IndexFlatIP',
    'BM25'                   : 'BM25, busca léxica, keyword search, TF-IDF, frequência de termos',
}

def enriquecer_chunk(texto, modulo='', arquivo=''):
    prefixo  = f'[{modulo}' + (f' / {arquivo}' if arquivo else '') + '] ' if modulo else ''
    sinonimos = [v for k, v in ENRIQUECIMENTO.items() if k.lower() in texto.lower()]
    resultado = prefixo + texto
    if sinonimos:
        resultado += ' | ' + '; '.join(sinonimos)
    return resultado

def chunk_por_secao(texto):
    chunks, titulo, linhas = [], 'Introdução', []
    for linha in texto.split('\n'):
        if linha.startswith('#'):
            if linhas:
                c = ' '.join(linhas).strip()
                if c: chunks.append({'titulo': titulo, 'conteudo': c})
            titulo, linhas = linha.lstrip('#').strip(), []
        elif linha.strip():
            linhas.append(linha.strip())
    if linhas:
        c = ' '.join(linhas).strip()
        if c: chunks.append({'titulo': titulo, 'conteudo': c})
    return chunks

def processar_doc(conteudo: str, modulo: str) -> list:
    """Chunking de AGENT_CONTEXT.md — mesmo padrão dos notebooks anteriores."""
    chunks = []
    for s in chunk_por_secao(conteudo):
        resumo = ' '.join(s['conteudo'].split()[:40])
        chunks.append({
            'chunk_busca'   : enriquecer_chunk(f"{s['titulo']}: {resumo}", modulo=modulo),
            'chunk_contexto': f"[{modulo} — {s['titulo']}]\n{s['conteudo']}",
            'titulo'        : s['titulo'],
            'modulo'        : modulo,
            'tipo'          : 'doc',
            'tipo_chunk'    : 'seção',
            'arquivo'       : 'AGENT_CONTEXT.md',
            'assinatura'    : '',
            'docstring'     : '',
            'linhas'        : (0, 0),
        })
    return chunks


# ── Coleta arquivos Python do EAI_07 ───────────────────────────────────────
def encontrar_pythons_eai07(pasta: str) -> list:
    """Retorna todos os .py do EAI_07, ignorando pastas de cache e env."""
    ignorar = {'.git', 'venv', '.venv', '__pycache__', 'node_modules', '.ipynb_checkpoints'}
    result  = []
    for raiz, dirs, arquivos in os.walk(pasta):
        dirs[:] = [d for d in dirs if d not in ignorar and not d.startswith('.')]
        for arq in arquivos:
            if arq.endswith('.py'):
                result.append(os.path.join(raiz, arq))
    return sorted(result)

def encontrar_agent_contexts(pasta_raiz: str) -> list:
    encontrados, ignorar = [], {'.git', 'venv', '.venv', '__pycache__'}
    for raiz, dirs, arquivos in os.walk(pasta_raiz):
        dirs[:] = [d for d in dirs if d not in ignorar and not d.startswith('.')]
        if 'AGENT_CONTEXT.md' in arquivos:
            partes = raiz.replace('\\', '/').split('/')
            modulo = next((p for p in partes if p.startswith('EAI_')), os.path.basename(raiz))
            encontrados.append((modulo, os.path.join(raiz, 'AGENT_CONTEXT.md')))
    return sorted(encontrados)

print('Funções de coleta definidas.')

Funções de coleta definidas.


In [5]:
# ── Constrói o índice híbrido ──────────────────────────────────────────────
CACHE_PATH = '../data/cache/indice_codigo.pkl'
os.makedirs(os.path.dirname(CACHE_PATH), exist_ok=True)

def construir_indice_hibrido(todos_chunks: list) -> dict:
    textos_busca = [c['chunk_busca'] for c in todos_chunks]
    print(f'Gerando embeddings para {len(textos_busca)} chunks...')
    t0   = time.time()
    embs = modelo_emb.encode(
        textos_busca, normalize_embeddings=True,
        show_progress_bar=True, batch_size=64
    ).astype(np.float32)
    idx = faiss.IndexFlatIP(embs.shape[1])
    idx.add(embs)
    print(f'Indexado em {time.time()-t0:.1f}s')
    return {'faiss': idx, 'chunks': todos_chunks, 'embs': embs}

def salvar_cache(indice_dict, caminho):
    dados = {
        'faiss_bytes': faiss.serialize_index(indice_dict['faiss']),
        'chunks'     : indice_dict['chunks'],
        'embs'       : indice_dict['embs'],
    }
    with open(caminho, 'wb') as f:
        pickle.dump(dados, f)
    print(f"Cache salvo: {caminho} ({os.path.getsize(caminho)/1024/1024:.1f} MB)")

def carregar_cache(caminho):
    with open(caminho, 'rb') as f:
        dados = pickle.load(f)
    return {
        'faiss' : faiss.deserialize_index(dados['faiss_bytes']),
        'chunks': dados['chunks'],
        'embs'  : dados['embs'],
    }


if os.path.exists(CACHE_PATH):
    print('Cache encontrado! Carregando...')
    t0     = time.time()
    INDICE = carregar_cache(CACHE_PATH)
    print(f'Carregado em {time.time()-t0:.2f}s ({len(INDICE["chunks"])} chunks)')
else:
    todos_chunks = []

    # 1) Código: .py do EAI_07
    pys = encontrar_pythons_eai07(EAI07_BASE)
    print(f'Arquivos .py encontrados: {len(pys)}')
    for py in pys:
        nome = os.path.relpath(py, EAI07_BASE).replace('\\', '/')
        cs   = extrair_chunks_ast(py, 'EAI_07_AI_Generative')
        print(f'  {nome}: {len(cs)} chunks')
        todos_chunks.extend(cs)

    # 2) Docs: AGENT_CONTEXT.md de todos os módulos
    acs = encontrar_agent_contexts(PROJETO_BASE)
    print(f'\nAgent contexts encontrados: {len(acs)}')
    for modulo, caminho in acs:
        with open(caminho, 'r', encoding='utf-8') as f:
            conteudo = f.read()
        cs = processar_doc(conteudo, modulo)
        todos_chunks.extend(cs)

    # Estatísticas
    n_codigo = sum(1 for c in todos_chunks if c['tipo'] == 'codigo')
    n_doc    = sum(1 for c in todos_chunks if c['tipo'] == 'doc')
    print(f'\nTotal chunks: {len(todos_chunks)} ({n_codigo} código + {n_doc} doc)')

    INDICE = construir_indice_hibrido(todos_chunks)
    salvar_cache(INDICE, CACHE_PATH)

# Resumo do índice
n_cod = sum(1 for c in INDICE['chunks'] if c['tipo'] == 'codigo')
n_doc = sum(1 for c in INDICE['chunks'] if c['tipo'] == 'doc')
print(f'\nÍndice pronto: {INDICE["faiss"].ntotal} chunks ({n_cod} código + {n_doc} doc)')

Arquivos .py encontrados: 4
  agent.py: 5 chunks
  shared/llm_factory.py: 10 chunks
  shared/tool_runner.py: 13 chunks
  teste_deepseek.py: 1 chunks

Agent contexts encontrados: 26

Total chunks: 1582 (29 código + 1553 doc)
Gerando embeddings para 1582 chunks...


Batches:   0%|          | 0/25 [00:00<?, ?it/s]

Indexado em 77.1s
Cache salvo: ../data/cache/indice_codigo.pkl (5.5 MB)

Índice pronto: 1582 chunks (29 código + 1553 doc)


---
## 3. Busca Híbrida: Semântica + BM25

### Por que BM25?

Embedding semântico é ótimo para intenção, mas pode perder nomes exatos.  
Exemplo: buscar `"chat_stream"` — o embedding pode trazer coisas semanticamente relacionadas ("fluxo de texto", "geração contínua") mas perder a função exata com esse nome.

**BM25** (Best Match 25) é o algoritmo clássico de busca por keywords — o mesmo que motores de busca usavam antes de LLMs. Pontuação baseada em frequência de termos (TF) penalizada por comprimento do documento (IDF).

### Score combinado

```
score_final = α × score_semântico + (1 - α) × score_bm25
```

- `α = 1.0` → só semântico (comportamento do notebook 03)
- `α = 0.0` → só BM25 (busca léxica pura)
- `α = 0.7` → padrão recomendado para código

In [6]:
# ── Implementação BM25 from scratch (sem dependências extras) ──────────────

class BM25:
    """
    BM25 (Best Match 25) — algoritmo clássico de busca léxica.
    Implementado do zero para não adicionar dependências ao ambiente.

    Parâmetros padrão recomendados pela literatura:
      k1 = 1.5  → saturação de frequência de termos
      b  = 0.75 → penalidade por comprimento do documento
    """

    def __init__(self, corpus: list[str], k1: float = 1.5, b: float = 0.75):
        self.k1     = k1
        self.b      = b
        self.corpus = corpus
        self._tokenizar_e_indexar()

    def _tokenizar(self, texto: str) -> list[str]:
        """Tokenização simples: lowercase + split por não-alfanumérico."""
        return re.findall(r'[a-záéíóúâêîôûãõàü\w]+', texto.lower())

    def _tokenizar_e_indexar(self):
        self.docs_tok   = [self._tokenizar(d) for d in self.corpus]
        self.N          = len(self.docs_tok)
        self.avgdl      = sum(len(d) for d in self.docs_tok) / self.N if self.N else 1
        # IDF: quantos documentos contêm cada termo
        df = Counter()
        for doc in self.docs_tok:
            for term in set(doc):
                df[term] += 1
        self.idf = {
            t: math.log((self.N - n + 0.5) / (n + 0.5) + 1)
            for t, n in df.items()
        }

    def score(self, query: str, idx: int) -> float:
        """Calcula BM25 score de um único documento."""
        doc    = self.docs_tok[idx]
        dl     = len(doc)
        tf_doc = Counter(doc)
        escore = 0.0
        for term in self._tokenizar(query):
            if term not in self.idf:
                continue
            tf  = tf_doc[term]
            num = tf * (self.k1 + 1)
            den = tf + self.k1 * (1 - self.b + self.b * dl / self.avgdl)
            escore += self.idf[term] * num / den
        return escore

    def buscar(self, query: str, top_k: int = 10) -> list[tuple[int, float]]:
        """Retorna (índice, score) dos top_k documentos mais relevantes."""
        scores = [(i, self.score(query, i)) for i in range(self.N)]
        scores.sort(key=lambda x: x[1], reverse=True)
        return scores[:top_k]


# Constrói o índice BM25 sobre os chunk_busca
print('Construindo índice BM25...')
t0 = time.time()
corpus_bm25 = [c['chunk_busca'] for c in INDICE['chunks']]
BM25_IDX    = BM25(corpus_bm25)
print(f'BM25 pronto em {time.time()-t0:.2f}s ({BM25_IDX.N} documentos)')

Construindo índice BM25...
BM25 pronto em 0.11s (1582 documentos)


In [7]:
# ── Busca híbrida: combina semântico + BM25 ────────────────────────────────

def buscar_hibrido(
    query      : str,
    top_k      : int   = 5,
    alfa       : float = 0.7,
    filtro_tipo: str   = None,   # 'codigo' | 'doc' | None
    score_min  : float = 0.0,
) -> list:
    """
    Busca híbrida combinando score semântico (FAISS) e léxico (BM25).

    alfa: peso do score semântico (0.0 a 1.0)
      - alfa alto  → prioriza intenção / similaridade conceitual
      - alfa baixo → prioriza nomes exatos / keywords
    """
    n_total = len(INDICE['chunks'])

    # ── Score semântico (FAISS) ────────────────────────────────────────────
    emb_q          = modelo_emb.encode([query], normalize_embeddings=True).astype(np.float32)
    k_faiss        = min(n_total, max(top_k * 5, 50))
    scores_f, pos  = INDICE['faiss'].search(emb_q, k_faiss)
    sem_scores     = {int(pos[0][i]): float(scores_f[0][i]) for i in range(k_faiss)}

    # Normaliza semântico para [0, 1] (já é cosine similarity com normalize=True)
    max_sem = max(sem_scores.values()) if sem_scores else 1.0
    sem_norm = {i: s / max_sem for i, s in sem_scores.items()}

    # ── Score BM25 ────────────────────────────────────────────────────────
    bm25_raw    = BM25_IDX.buscar(query, top_k=k_faiss)
    bm25_scores = {i: s for i, s in bm25_raw}

    # Normaliza BM25 para [0, 1]
    max_bm = max(bm25_scores.values()) if bm25_scores else 1.0
    bm25_norm = {i: s / max_bm for i, s in bm25_scores.items()} if max_bm > 0 else {}

    # ── Score combinado ───────────────────────────────────────────────────
    candidatos = set(sem_norm) | set(bm25_norm)
    combined   = {
        i: alfa * sem_norm.get(i, 0.0) + (1 - alfa) * bm25_norm.get(i, 0.0)
        for i in candidatos
    }

    # Ordena e filtra
    ranking = sorted(combined.items(), key=lambda x: x[1], reverse=True)

    resultados = []
    for idx, score in ranking:
        if len(resultados) >= top_k:
            break
        chunk = INDICE['chunks'][idx]
        if filtro_tipo and chunk['tipo'] != filtro_tipo:
            continue
        if score < score_min:
            continue
        resultados.append({
            'contexto'    : chunk['chunk_contexto'],
            'score'       : round(score, 4),
            'score_sem'   : round(sem_norm.get(idx, 0.0), 4),
            'score_bm25'  : round(bm25_norm.get(idx, 0.0), 4),
            'meta': {
                'modulo'    : chunk['modulo'],
                'titulo'    : chunk['titulo'],
                'tipo'      : chunk['tipo'],
                'tipo_chunk': chunk['tipo_chunk'],
                'arquivo'   : chunk['arquivo'],
            }
        })

    return resultados


print('buscar_hibrido() definida.')

buscar_hibrido() definida.


In [8]:
# ── Demonstração: o poder da busca híbrida ────────────────────────────────
#
# Query com nome exato de função — BM25 deve ajudar o semântico

query = 'como funciona chat_stream no llm_factory?'

print(f'Query: "{query}"')
print(f'{"-"*60}')

for alfa, label in [(1.0, 'Só semântico (α=1.0)'), (0.0, 'Só BM25 (α=0.0)'), (0.7, 'Híbrido (α=0.7)')]:
    resultados = buscar_hibrido(query, top_k=3, alfa=alfa)
    print(f'\n── {label} {"-"*(40-len(label))}')
    for r in resultados:
        m = r['meta']
        print(f"  [{r['score']:.3f}] sem={r['score_sem']:.3f} bm25={r['score_bm25']:.3f} "
              f"| {m['tipo']:6} | {m['titulo']:25} | {m['arquivo']}")

Query: "como funciona chat_stream no llm_factory?"
------------------------------------------------------------

── Só semântico (α=1.0) --------------------
  [1.000] sem=1.000 bm25=0.806 | doc    | PERGUNTAS FREQUENTES      | AGENT_CONTEXT.md
  [0.976] sem=0.976 bm25=0.570 | codigo | chat                      | EAI_07_AI_Generative/shared/llm_factory.py
  [0.872] sem=0.872 bm25=0.992 | codigo | chat_stream               | EAI_07_AI_Generative/shared/llm_factory.py

── Só BM25 (α=0.0) -------------------------
  [1.000] sem=0.783 bm25=1.000 | doc    | Verificar provider ativo  | AGENT_CONTEXT.md
  [0.992] sem=0.872 bm25=0.992 | codigo | chat_stream               | EAI_07_AI_Generative/shared/llm_factory.py
  [0.985] sem=0.000 bm25=0.985 | doc    | Regularização Detalhada   | AGENT_CONTEXT.md

── Híbrido (α=0.7) -------------------------
  [0.942] sem=1.000 bm25=0.806 | doc    | PERGUNTAS FREQUENTES      | AGENT_CONTEXT.md
  [0.908] sem=0.872 bm25=0.992 | codigo | chat_stream          

---
## 4. Assistente de Código

O assistente usa a busca híbrida para responder perguntas sobre o próprio projeto.  
O contexto enviado ao LLM mistura código real (com assinaturas, docstrings) e documentação dos módulos.

In [9]:
SYSTEM_PROMPT = """\
Você é o Assistente de Código do projeto ESPECIALISTA_EM_IA de Carlos Henrique.
Responde perguntas sobre o código e a arquitetura do projeto usando o contexto fornecido.

Diretrizes:
- Ao citar código, use blocos ```python com o trecho relevante
- Mencione o arquivo de origem quando relevante (ex: shared/llm_factory.py)
- Se o contexto tiver tanto código quanto docs sobre o mesmo tópico, integre os dois
- Se a resposta não estiver no contexto, diga claramente
- Seja preciso e técnico — o usuário é desenvolvedor
"""

class AssistenteCodigoRAG:
    def __init__(self, alfa: float = 0.7, top_k: int = 5):
        self.alfa      = alfa
        self.top_k     = top_k
        self.historico = []

    def responder(self, pergunta: str, filtro_tipo: str = None, verbose: bool = False) -> str:
        # Busca híbrida
        resultados = buscar_hibrido(
            pergunta, top_k=self.top_k, alfa=self.alfa,
            filtro_tipo=filtro_tipo
        )

        if verbose:
            print(f'Query busca: {pergunta[:60]}...')
            print(f'Chunks: {len(resultados)}')
            for r in resultados:
                m = r['meta']
                print(f"  [{r['score']:.3f}] {m['tipo']:6} | {m['titulo']:25} | {m['arquivo']}")
            print()

        contexto = '\n\n---\n\n'.join(r['contexto'] for r in resultados)
        prompt   = f'Contexto recuperado:\n\n{contexto}\n\nPergunta: {pergunta}'

        self.historico.append({'role': 'user', 'content': prompt})

        resp = llm.chat.completions.create(
            model    = LLM_MODEL,
            messages = [{'role': 'system', 'content': SYSTEM_PROMPT}] + self.historico,
        )
        resposta = resp.choices[0].message.content
        self.historico.append({'role': 'assistant', 'content': resposta})
        return resposta

    def resetar(self):
        self.historico = []


print('AssistenteCodigoRAG definido.')

AssistenteCodigoRAG definido.


In [10]:
# ── Teste 1: pergunta sobre código específico ─────────────────────────────
assistente = AssistenteCodigoRAG(alfa=0.7)

perguntas = [
    'Quais funções estão disponíveis no llm_factory e o que cada uma faz?',
    'Como o tool_runner lida com respostas do DeepSeek que não seguem o formato padrão de tools?',
]

for p in perguntas:
    print(f'\n👤 {p}')
    print('─' * 55)
    print(f'🤖 {assistente.responder(p, verbose=True)}')


👤 Quais funções estão disponíveis no llm_factory e o que cada uma faz?
───────────────────────────────────────────────────────
Query busca: Quais funções estão disponíveis no llm_factory e o que cada ...
Chunks: 5
  [0.988] codigo | chat                      | EAI_07_AI_Generative/shared/llm_factory.py
  [0.895] doc    | PERGUNTAS FREQUENTES      | AGENT_CONTEXT.md
  [0.764] doc    | O Que o Modelo REALMENTE Faz? | AGENT_CONTEXT.md
  [0.760] doc    | CONCEITOS — REFERÊNCIA RÁPIDA | AGENT_CONTEXT.md
  [0.750] codigo | get_provider_info         | EAI_07_AI_Generative/shared/llm_factory.py

🤖 Com base no contexto fornecido, aqui estão as funções disponíveis no `shared/llm_factory.py` e suas funcionalidades:

## 1. `chat()` - Função principal para interação com LLMs
```python
def chat(
    prompt: str,
    system: Optional[str] = None,
    temperature: Optional[float] = None,
    max_tokens: Optional[int] = None,
) -> str:
```
**Função**: Envia uma mensagem para o LLM configurado no `.env

In [11]:
# ── Teste 2: filtro por tipo — apenas código ──────────────────────────────
assistente2 = AssistenteCodigoRAG(alfa=0.6)

print('👤 Como os embeddings são gerados e indexados no RAG básico?')
print('   (filtro: apenas chunks de código)\n')
print('─' * 55)
resp = assistente2.responder(
    'Como os embeddings são gerados e indexados?',
    filtro_tipo='codigo',
    verbose=True
)
print(f'🤖 {resp}')

👤 Como os embeddings são gerados e indexados no RAG básico?
   (filtro: apenas chunks de código)

───────────────────────────────────────────────────────
Query busca: Como os embeddings são gerados e indexados?...
Chunks: 1
  [0.230] codigo | _adicionar_resultados     | EAI_07_AI_Generative/shared/tool_runner.py

🤖 Com base no contexto fornecido, não há informações sobre como os embeddings são gerados e indexados. O contexto mostra apenas o arquivo `tool_runner.py` e a função `_adicionar_resultados`, que trata da execução de funções/tools e adição dos resultados ao histórico de mensagens.

Para responder sobre geração e indexação de embeddings, seria necessário contexto sobre:
- Módulos de processamento de texto/documentos
- Uso de modelos de embedding (como OpenAI, Hugging Face, etc.)
- Sistemas de indexação (FAISS, Chroma, Pinecone, etc.)
- Funções específicas para criação e armazenamento de vetores

O contexto atual foca apenas na execução de tools/funções dentro do fluxo de convers

In [12]:
# ── Teste 3: pergunta mista — código + docs integrados ────────────────────
assistente3 = AssistenteCodigoRAG(alfa=0.7, top_k=6)

print('👤 Como funciona o sistema de RAG do EAI_07? Descreva a arquitetura.')
print('─' * 55)
print(f'🤖 {assistente3.responder("Como funciona o sistema de RAG do EAI_07? Descreva a arquitetura.", verbose=True)}')

👤 Como funciona o sistema de RAG do EAI_07? Descreva a arquitetura.
───────────────────────────────────────────────────────
Query busca: Como funciona o sistema de RAG do EAI_07? Descreva a arquite...
Chunks: 6
  [0.916] doc    | PERGUNTAS FREQUENTES      | AGENT_CONTEXT.md
  [0.737] doc    | Arquitetura               | AGENT_CONTEXT.md
  [0.700] doc    | Conceito                  | AGENT_CONTEXT.md
  [0.682] doc    | Estrutura Pedagógica      | AGENT_CONTEXT.md
  [0.677] doc    | 3. Textos Muito Curtos    | AGENT_CONTEXT.md
  [0.674] doc    | Palavras-chave com alto peso | AGENT_CONTEXT.md

🤖 Com base no contexto fornecido, **não há informações sobre o sistema de RAG (Retrieval-Augmented Generation) do EAI_07**.

O contexto recuperado aborda principalmente:
1.  **EAI_07_AI_Generative**: Focado em perguntas frequentes sobre o uso de LLMs (como trocar de provider, formatos de tool calls, streaming).
2.  **EAI_04_NLP_Classico**: Focado em arquitetura de classificação dupla, conceitos de 

---
## Resumo

| Técnica | Problema que resolve |
|---|---|
| **Chunking por AST** | Cortes no meio de funções quebram contexto e semântica do código |
| **Metadados ricos** | Assinatura, docstring, arquivo e linhas ficam disponíveis para o LLM |
| **Índice híbrido** | Código e docs falam do mesmo conceito — faz sentido indexá-los juntos |
| **BM25** | Nomes exatos (`chat_stream`, `llm_factory`) que embedding semântico perde |
| **Score combinado (α)** | Ajuste fino: mais semântico para intenção, mais BM25 para nomes exatos |
| **Filtro por `tipo`** | Isola busca só em código ou só em docs quando necessário |

### Parâmetro `α` — guia rápido

| Caso de uso | α recomendado |
|---|---|
| Busca por nome de função/variável | 0.4 – 0.5 |
| Busca por conceito ("como funciona X") | 0.8 – 1.0 |
| Uso geral em código | 0.6 – 0.7 |
| Documentação técnica | 0.7 – 0.8 |